# 02. YouTube 데이터 수집

YouTube Data API v3를 사용하여 `YT_ChannelData_2026-03-28_clean.csv`의 채널 ID를 기반으로  
영상 메타데이터와 썸네일을 수집합니다.

## 수집 흐름

```
채널 목록 (YT_ChannelData_2026-03-28_clean.csv)
    ↓
채널별 업로드 재생목록 조회
    ↓
영상 상세 정보 수집 (snippet, contentDetails, statistics...)
    ↓
Shorts / 라이브 필터링
    ↓
썸네일 다운로드 → thumbnails/{channel_name}/{video_id}.jpg
    ↓
YT_dataset_v1.csv 저장
```

> **API 키는 `.env` 파일에 저장합니다.** `.env`는 `.gitignore`에 포함되어 있어 레포에 올라가지 않습니다.

---
## 패키지 설치

In [15]:
%pip install google-api-python-client python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


---
## API 키 설정

프로젝트 루트의 `.env` 파일에 아래 형식으로 API 키를 저장하세요:

```
YOUTUBE_API_KEY=여기에_발급받은_키_입력
```

- API 키 발급: [Google Cloud Console](https://console.cloud.google.com/) → API 및 서비스 → YouTube Data API v3 활성화
- `.env` 파일은 `.gitignore`에 포함되어 있어 **절대 레포에 올라가지 않습니다**

In [16]:
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError(".env 파일에 YOUTUBE_API_KEY가 설정되지 않았습니다.")
print("API 키 로드 완료")

API 키 로드 완료


---
## 라이브러리 임포트 & 전역 설정

In [17]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
import requests
import time
import os
from pathlib import Path

# ── 설정 ──────────────────────────────────────────────────────────────────
CHANNELS_CSV   = Path("YT_ChannelData_2026-03-28_clean.csv")  # 채널 목록
OUTPUT_CSV     = Path("YT_dataset_v1.csv")                    # 결과 저장 경로
THUMBNAIL_DIR  = Path("thumbnails")                # 썸네일 저장 루트
VIDEO_COUNT    = 1                                 # 채널당 수집할 최대 영상 수
REQUEST_DELAY  = 0.3                               # API 호출 간 대기(초)

print(f"채널 목록: {CHANNELS_CSV}")
print(f"결과 파일: {OUTPUT_CSV}")
print(f"채널당 영상 수: {VIDEO_COUNT}")

채널 목록: YT_ChannelData_2026-03-28_clean.csv
결과 파일: YT_dataset_v1.csv
채널당 영상 수: 1


---
## 채널 ID 조회

`@핸들명`으로 채널 ID를 검색합니다. `YT_channelsList_v1.csv`에 이미 `channel_id`가 있으므로 보통 직접 사용하지 않아도 됩니다.  
새 채널을 추가할 때 ID를 모르는 경우에 활용합니다.

In [18]:
def get_channel_id_from_handle(api_key: str, handle: str) -> str | None:
    """
    YouTube 채널 핸들(@포함 또는 미포함)로 channel_id를 조회합니다.
    예: get_channel_id_from_handle(API_KEY, "@재활의학과탑팀")
    """
    youtube = build("youtube", "v3", developerKey=api_key)
    clean_handle = handle.replace("@", "")
    response = youtube.channels().list(
        part="id",
        forHandle=clean_handle,
    ).execute()

    if response.get("items"):
        return response["items"][0]["id"]
    return None


# ── 사용 예시 ──────────────────────────────────────────────────────────────
# ch_id = get_channel_id_from_handle(API_KEY, "@재활의학과탑팀")
# print(ch_id)  # → UCWW--vYvkR604lbGg92J2_A
print("get_channel_id_from_handle 함수 정의 완료")

get_channel_id_from_handle 함수 정의 완료


---
## 헬퍼 함수

### Shorts 판별 방식

`/shorts/{video_id}` URL로 요청했을 때:
- **200 반환** → Shorts 영상
- **303 리다이렉트** → 일반 영상

duration 기준보다 정확하지만 영상당 HTTP 요청 1회가 추가됩니다.

In [19]:
def _is_shorts(video_id: str, session: requests.Session) -> bool:
    """/shorts/{id}가 리다이렉트 없이 200이면 Shorts."""
    url = f"https://www.youtube.com/shorts/{video_id}"
    try:
        resp = session.get(url, allow_redirects=False, timeout=8)
        return resp.status_code == 200
    except requests.RequestException:
        return False


def _is_live(snippet: dict) -> bool:
    """라이브 방송 또는 예정된 방송 여부."""
    return snippet.get("liveBroadcastContent") in ("live", "upcoming")


def _fetch_video_details(youtube, video_ids: list) -> list:
    """video_id 목록으로 영상 상세 정보를 일괄 조회합니다."""
    resp = youtube.videos().list(
        id=",".join(video_ids),
        part="snippet,contentDetails,statistics,status,topicDetails",
    ).execute()
    return resp.get("items", [])


print("헬퍼 함수 정의 완료")

헬퍼 함수 정의 완료


---
## 메인 수집 함수

채널별로 아래 순서로 처리합니다:

1. 채널 업로드 재생목록 ID 조회
2. 재생목록에서 영상 ID 페이지 순회
3. 영상별 상세 정보 조회
4. 라이브 → 스킵 / Shorts → 스킵
5. 썸네일 다운로드
6. `video_count`개 채우면 다음 채널로

> 중간에 중단됐을 경우 `OUTPUT_CSV`가 이미 있다면 이어쓰기를 지원합니다.

In [20]:
def youtube_data_extract(
    api_key: str,
    channels_csv: Path = CHANNELS_CSV,
    output_csv: Path = OUTPUT_CSV,
    video_count: int = VIDEO_COUNT,
    delay: float = REQUEST_DELAY,
) -> pd.DataFrame:

    youtube = build("youtube", "v3", developerKey=api_key)
    channels_df = pd.read_csv(channels_csv)
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    # 이미 수집된 video_id가 있으면 건너뜀 (이어쓰기)
    done_ids: set = set()
    if output_csv.exists():
        existing = pd.read_csv(output_csv, usecols=["video_id"])
        done_ids = set(existing["video_id"].astype(str))
        print(f"기존 데이터 로드: {len(done_ids)}개 영상 (이어쓰기 모드)")

    all_data: list = []

    for _, row in channels_df.iterrows():
        channel_name = row["channel_name"]
        channel_id   = row["channel_id"]
        print(f"\n처리 중: {channel_name} ({channel_id})")

        # 1. 채널 업로드 재생목록 ID 조회
        ch_resp = youtube.channels().list(
            id=channel_id, part="contentDetails"
        ).execute()
        if not ch_resp.get("items"):
            print(f"  → 채널 없음, 스킵")
            continue
        uploads_id = ch_resp["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

        # 썸네일 저장 폴더 (채널명 특수문자 제거)
        safe_name = "".join(c for c in channel_name if c.isalnum() or c in ("_", "-"))
        thumb_dir = THUMBNAIL_DIR / safe_name
        thumb_dir.mkdir(parents=True, exist_ok=True)

        saved, skipped_shorts, skipped_live, skipped_done = 0, 0, 0, 0
        next_page_token = None

        while saved < video_count:
            # 2. 재생목록 페이지 조회
            pl_kwargs = dict(
                playlistId=uploads_id,
                part="snippet",
                maxResults=min((video_count - saved) * 2, 50),
            )
            if next_page_token:
                pl_kwargs["pageToken"] = next_page_token

            try:
                pl_resp = youtube.playlistItems().list(**pl_kwargs).execute()
            except HttpError as e:
                print(f"  → 재생목록 조회 실패 (스킵): {e.reason}")
                break

            page_ids = [
                item["snippet"]["resourceId"]["videoId"]
                for item in pl_resp.get("items", [])
            ]
            if not page_ids:
                break

            # 3. 영상 상세 정보 조회
            for item in _fetch_video_details(youtube, page_ids):
                if saved >= video_count:
                    break

                v_id    = item["id"]
                snippet = item.get("snippet", {})
                content = item.get("contentDetails", {})
                stats   = item.get("statistics", {})
                status  = item.get("status", {})
                topics  = item.get("topicDetails", {})

                # 이미 수집된 영상 스킵
                if v_id in done_ids:
                    skipped_done += 1
                    continue

                # 라이브 스킵
                if _is_live(snippet):
                    skipped_live += 1
                    continue

                # Shorts 스킵 (URL 리다이렉트 방식)
                time.sleep(delay)
                if _is_shorts(v_id, session):
                    skipped_shorts += 1
                    continue

                # 4. 썸네일 다운로드 (maxres → high 순)
                thumbnails = snippet.get("thumbnails", {})
                thumb_info = thumbnails.get("maxres") or thumbnails.get("high") or {}
                thumb_url  = thumb_info.get("url", "")
                local_path = ""

                if thumb_url:
                    local_path = str(thumb_dir / f"{v_id}.jpg")
                    try:
                        img_data = requests.get(thumb_url, timeout=10).content
                        with open(local_path, "wb") as f:
                            f.write(img_data)
                    except Exception as e:
                        print(f"  → 썸네일 실패 ({v_id}): {e}")
                        local_path = ""

                all_data.append({
                    "channel_name":     channel_name,
                    "channel_id":       channel_id,
                    "video_id":         v_id,
                    "title":            snippet.get("title"),
                    "description":      snippet.get("description"),
                    "published_at":     snippet.get("publishedAt"),
                    "tags":             ",".join(snippet.get("tags", [])),
                    "category_id":      snippet.get("categoryId"),
                    "default_language": snippet.get("defaultLanguage", "N/A"),
                    "duration":         content.get("duration"),
                    "dimension":        content.get("dimension"),
                    "definition":       content.get("definition"),
                    "caption":          content.get("caption"),
                    "view_count":       stats.get("viewCount", 0),
                    "like_count":       stats.get("likeCount", 0),
                    "favorite_count":   stats.get("favoriteCount", 0),
                    "comment_count":    stats.get("commentCount", 0),
                    "privacy_status":   status.get("privacyStatus"),
                    "license":          status.get("license"),
                    "embeddable":       status.get("embeddable"),
                    "made_for_kids":    status.get("madeForKids"),
                    "topic_categories": ",".join(topics.get("topicCategories", [])),
                    "thumbnail_url":    thumb_url,
                    "thumbnail_path":   local_path,
                })
                done_ids.add(v_id)
                saved += 1

            next_page_token = pl_resp.get("nextPageToken")
            if not next_page_token:
                break

        print(f"  → 저장: {saved}개 | 스킵: Shorts {skipped_shorts} / 라이브 {skipped_live} / 기수집 {skipped_done}")

    # 5. CSV 저장 (기존 데이터와 합치기)
    new_df = pd.DataFrame(all_data)
    if output_csv.exists() and not new_df.empty:
        existing_df = pd.read_csv(output_csv, encoding="utf-8-sig")
        final_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        final_df = new_df

    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n저장 완료: 총 {len(final_df)}개 영상 → {output_csv}")
    return final_df


print("youtube_data_extract 함수 정의 완료")

youtube_data_extract 함수 정의 완료


---
## 실행

`VIDEO_COUNT`를 줄이면 테스트 속도가 빠릅니다 (예: `5`).  
API 할당량은 하루 **10,000 단위**이며, 영상 1개당 약 5~10 단위 소비됩니다.

In [21]:
df = youtube_data_extract(
    api_key=API_KEY,
    channels_csv=CHANNELS_CSV,
    output_csv=OUTPUT_CSV,
    video_count=VIDEO_COUNT,
)

print(df[["channel_name", "title", "duration", "view_count", "thumbnail_path"]].head(10))


처리 중: 다이어트 과학자 최겸 Gyum Choi (UChXDq9Izwq-pTzeUWzwXL3A)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 한의사트레이너 (UC_KjTyC5e0DFNxn6GIgBxcQ)
  → 저장: 1개 | 스킵: Shorts 2 / 라이브 0 / 기수집 0

처리 중: 블락스blocks (UC5qF47U0ErUHBgK4Rgy5ymA)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 김지만TV (UCCFD5pLnLB2oEb4tPm1YemA)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 김소영건강체조티비-Health gymnastics (UCSi2gkWNFwoMpb5Nu19EnpA)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 통증교정 수연쌤 (UCafAiwzaq1wXHIJOCm4zSoQ)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 우수한 영양학 (UCxkwgIYCm3vLlYITOzdByZg)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 빵느 (UCRrZ5RYIalHLiHq5ftzxM6A)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 홍삼아 댄스&에어로빅 (UCNcl4-NKaNrX0CBK3Gmd5Ag)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 약초1번지 (UCvf59uLGe-VyEcoNsp15K0w)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: DENTIS_official (UCv5oPNeMFM1TY4RPv8DFpiw)
  → 저장: 1개 | 스킵: Shorts 0 / 라이브 0 / 기수집 0

처리 중: 보이스멘토소리방송 (UCkJThHTYxa9g_NMud

---
## 결과 확인

In [22]:
df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")

print(f"총 영상 수    : {len(df)}")
print(f"채널 수       : {df['channel_name'].nunique()}")
print(f"썸네일 있음   : {df['thumbnail_path'].notna().sum()} / {len(df)}")
print()
print(df.groupby("channel_name").size().rename("영상 수").to_string())

총 영상 수    : 989
채널 수       : 788
썸네일 있음   : 988 / 989

channel_name
 건강한tv                                        1
 시니어건강                                        2
 요가배 | 뇌과학 치유요가&마인드                           1
1분도수 | 목 전문 도수치료사                             1
1분운동프로젝트                                      1
1분팁                                           1
365노후건강                                       1
3분 건강 - 보약 같은 음식 소개                           1
50대채널                                         1
60세 이후 건강                                     1
8체질연구소                                        1
ADHD 대표채널                                     1
Basant                                        2
Beautiful Relaxing Piano                      1
Brain Quiz                                    1
CloudHospital TV                              1
Curistory                                     1
DENTIS_official                               1
DJ저요                                          2
DN.Beauty Spanish   